## 1. Imports e Dependências

In [1]:
import time
import tracemalloc
import functools
import folium

## 2. Fator de Horário

O peso de cada aresta é multiplicado por um fator que depende do horário de partida:

| Faixa | Fator | Motivo |
|---|---|---|
| 5h – 7h | × 0,6 | Bônus: metrô vazio |
| 7h – 9h | × 1,5 | Pico da manhã |
| 9h – 17h | × 1,0 | Horário normal |
| 17h – 20h | × 2,0 | Pico da tarde (penalidade) |
| Demais | × 1,0 | Madrugada/noite normal |


In [2]:
def fator_horario(hora: int) -> float:
    """Retorna o multiplicador de custo conforme a faixa horária."""
    if 5 <= hora < 7:    return 0.6   # bônus madrugada
    elif 7 <= hora < 9:  return 1.5   # pico manhã
    elif 9 <= hora < 17: return 1.0   # horário normal
    elif 17 <= hora < 20: return 2.0  # pico tarde — penalidade
    else:                 return 1.0  # demais horários

## 3. Modelagem do Grafo

Cada rede de metrô é representada como um **grafo não-dirigido ponderado**:
- **Nós** = estações
- **Arestas** = conexões entre estações adjacentes
- **Peso** = tempo estimado de viagem (minutos)

O grafo é **não-dirigido** pois os trens operam em via dupla (ida e volta) em todas as três redes.

A estrutura interna é um dicionário `{estação: [(vizinho, peso), ...]}`.


In [3]:
def construir_grafo(arestas: list) -> dict:
    """Constrói um grafo não-dirigido a partir de uma lista de arestas (origem, destino, peso)."""
    grafo = {}
    for origem, destino, peso in arestas:
        grafo.setdefault(origem, []).append((destino, peso))
        grafo.setdefault(destino, []).append((origem, peso))
    return grafo

## 4. São Paulo — Metrô + CPTM

**Origem:** Tucuruvi (Linha 1-Azul)  
**Destino:** Capão Redondo (Linha 5-Lilás)  
**Integração obrigatória:** Paulista (L2↔L4), Santa Cruz (L1↔L5), Chácara Klabin (L2↔L5), entre outras.

### Caminhos distintos possíveis
1. Tucuruvi → ... → Sé → Paraíso → Ana Rosa → Santa Cruz → ... → Capão Redondo
2. Tucuruvi → ... → Luz → República → Paulista → Faria Lima → ... → Santo Amaro → ... → Capão Redondo
3. Tucuruvi → ... → Sé → Anhangabaú → República → Paulista → ... (via Linha 4)


In [4]:
coordenadas_sp = {
    "Tucuruvi":               (-23.4803, -46.6037),
    "Parada Inglesa":         (-23.4869, -46.6087),
    "Jardim São Paulo":       (-23.4921, -46.6171),
    "Carandiru":              (-23.5089, -46.6249),
    "Santana":                (-23.5024, -46.6247),
    "Portuguesa-Tietê":       (-23.5159, -46.6252),
    "Armênia":                (-23.5254, -46.6292),
    "Tiradentes":             (-23.5307, -46.6321),
    "Luz":                    (-23.5385, -46.6360),
    "São Bento":              (-23.5444, -46.6341),
    "Sé":                     (-23.5493, -46.6335),
    "Japão-Liberdade":        (-23.5555, -46.6360),
    "Vergueiro":              (-23.5690, -46.6399),
    "Paraíso":                (-23.5759, -46.6414),
    "Ana Rosa":               (-23.5814, -46.6384),
    "Santa Cruz":             (-23.5992, -46.6367),
    "Chácara Klabin":         (-23.5925, -46.6302),
    "Vila Madalena":          (-23.5463, -46.6908),
    "Sumaré":                 (-23.5506, -46.6783),
    "Clínicas":               (-23.5542, -46.6707),
    "Consolação":             (-23.5561, -46.6624),
    "Paulista":               (-23.5550, -46.6621),
    "Trianon-Masp":           (-23.5625, -46.6552),
    "Brigadeiro":             (-23.5683, -46.6482),
    "República":              (-23.5431, -46.6439),
    "Anhangabaú":             (-23.5479, -46.6393),
    "Higienópolis-Mackenzie": (-23.5489, -46.6522),
    "Marechal Deodoro":       (-23.5350, -46.6545),
    "Santa Cecília":          (-23.5382, -46.6573),
    "Palmeiras-Barra Funda":  (-23.5271, -46.6597),
    "Faria Lima":             (-23.5676, -46.6928),
    "Pinheiros":              (-23.5659, -46.7011),
    "Butantã":                (-23.5650, -46.7069),
    "São Paulo-Morumbi":      (-23.5862, -46.7237),
    "Vila Sônia":             (-23.5915, -46.7356),
    "Santo Amaro":            (-23.6540, -46.7024),
    "Largo Treze":            (-23.6545, -46.7105),
    "Adolfo Pinheiro":        (-23.6500, -46.7041),
    "Alto da Boa Vista":      (-23.6414, -46.6989),
    "Borba Gato":             (-23.6331, -46.6928),
    "Brooklin":               (-23.6181, -46.6953),
    "Campo Belo":             (-23.6263, -46.6878),
    "Eucaliptos":             (-23.6099, -46.6683),
    "Moema":                  (-23.6032, -46.6620),
    "AACD-Servidor":          (-23.5974, -46.6523),
    "Hospital São Paulo":     (-23.5980, -46.6455),
    "Giovanni Gronchi":       (-23.6439, -46.7345),
    "Vila das Belezas":       (-23.6402, -46.7461),
    "Campo Limpo":            (-23.6490, -46.7588),
    "Capão Redondo":          (-23.6600, -46.7687),
}

arestas_sp = [
    # Linha 1 — Azul (Tucuruvi → Jabaquara, via Luz/Sé)
    ("Tucuruvi",          "Parada Inglesa",         3),
    ("Parada Inglesa",    "Jardim São Paulo",        3),
    ("Jardim São Paulo",  "Carandiru",              3),
    ("Carandiru",         "Santana",                3),
    ("Santana",           "Portuguesa-Tietê",       3),
    ("Portuguesa-Tietê",  "Armênia",                3),
    ("Armênia",           "Tiradentes",             3),
    ("Tiradentes",        "Luz",                    3),
    ("Luz",               "São Bento",              3),
    ("São Bento",         "Sé",                     3),
    ("Sé",                "Japão-Liberdade",        3),
    ("Japão-Liberdade",   "Vergueiro",              3),
    ("Vergueiro",         "Paraíso",                3),
    ("Paraíso",           "Ana Rosa",               3),
    ("Ana Rosa",          "Santa Cruz",             3),   # integração L1→L5
    # Linha 2 — Verde (Vila Madalena → Vila Prudente)
    ("Vila Madalena",     "Sumaré",                 3),
    ("Sumaré",            "Clínicas",               3),
    ("Clínicas",          "Consolação",             3),
    ("Consolação",        "Paulista",               3),   # integração L2↔L4
    ("Paulista",          "Trianon-Masp",           3),
    ("Trianon-Masp",      "Brigadeiro",             3),
    ("Brigadeiro",        "Paraíso",                3),   # integração L2↔L1
    ("Ana Rosa",          "Chácara Klabin",         3),   # integração L2↔L5
    # Linha 3 — Vermelha (Palmeiras-BF → Corinthians)
    ("Palmeiras-Barra Funda", "Marechal Deodoro",   3),
    ("Marechal Deodoro",  "Santa Cecília",          3),
    ("Santa Cecília",     "Higienópolis-Mackenzie", 3),
    ("Higienópolis-Mackenzie", "República",         3),
    ("República",         "Anhangabaú",             3),
    ("Anhangabaú",        "Sé",                     3),   # integração L3↔L1
    # Linha 4 — Amarela (Luz → Vila Sônia)
    ("Luz",               "Higienópolis-Mackenzie", 6),   # integração L4↔L3
    ("República",         "Paulista",               3),   # integração L4↔L2
    ("Paulista",          "Faria Lima",             3),
    ("Faria Lima",        "Pinheiros",              3),
    ("Pinheiros",         "Vila Madalena",          6),   # integração L4↔L2
    ("Pinheiros",         "Butantã",                3),
    ("Butantã",           "São Paulo-Morumbi",      3),
    ("São Paulo-Morumbi", "Vila Sônia",             3),
    # Linha 5 — Lilás (Capão Redondo → Chácara Klabin)
    ("Capão Redondo",     "Campo Limpo",            3),
    ("Campo Limpo",       "Vila das Belezas",       3),
    ("Vila das Belezas",  "Giovanni Gronchi",       3),
    ("Giovanni Gronchi",  "Santo Amaro",            3),
    ("Santo Amaro",       "Largo Treze",            3),
    ("Largo Treze",       "Adolfo Pinheiro",        3),
    ("Adolfo Pinheiro",   "Alto da Boa Vista",      3),
    ("Alto da Boa Vista", "Borba Gato",             3),
    ("Borba Gato",        "Brooklin",               3),
    ("Brooklin",          "Campo Belo",             3),
    ("Campo Belo",        "Eucaliptos",             3),
    ("Eucaliptos",        "Moema",                  3),
    ("Moema",             "AACD-Servidor",          3),
    ("AACD-Servidor",     "Hospital São Paulo",     3),
    ("Hospital São Paulo","Santa Cruz",             3),   # integração L5↔L1
    ("Santa Cruz",        "Chácara Klabin",         6),   # integração L5↔L2
]

grafo_sp = construir_grafo(arestas_sp)
print(f"Grafo SP: {len(grafo_sp)} estações, {sum(len(v) for v in grafo_sp.values())//2} arestas")

Grafo SP: 50 estações, 53 arestas


## 5. Beijing — Linhas 1, 2, 4 e 10

**Origem:** Sihui East (Linha 1)  
**Destino:** Xizhimen (Linhas 2 e 4)

### Caminhos distintos possíveis
1. L1 → Jianguomen → **L2** (via Chaoyangmen/Andingmen) → Xizhimen
2. L1 → Xidan → **L4** (via Xisi/Pinganli) → Xizhimen
3. L1 → Fuxingmen → **L2** (via Fuchengmen) → Xizhimen


In [5]:
coordenadas_bj = {
    # Linha 1
    "Sihui East":      (39.9043, 116.5169),
    "Sihui":           (39.9035, 116.5056),
    "Guomao":          (39.9085, 116.4614),
    "Yonganli":        (39.9083, 116.4483),
    "Jianguomen":      (39.9083, 116.4317),
    "Dongdan":         (39.9117, 116.4156),
    "Wangfujing":      (39.9136, 116.4083),
    "Tiananmen East":  (39.9083, 116.3978),
    "Tiananmen West":  (39.9083, 116.3897),
    "Xidan":           (39.9117, 116.3667),
    "Fuxingmen":       (39.9117, 116.3567),
    "Muxidi":          (39.9119, 116.3431),
    # Linha 2 (loop)
    "Chaoyangmen":     (39.9233, 116.4361),
    "Dongzhimen":      (39.9328, 116.4331),
    "Andingmen":       (39.9431, 116.4111),
    "Gulou Dajie":     (39.9450, 116.3953),
    "Jishuitan":       (39.9442, 116.3744),
    "Xizhimen":        (39.9431, 116.3483),
    "Fuchengmen":      (39.9233, 116.3578),
    # Linha 4
    "Xinjiekou":       (39.9331, 116.3617),
    "Pinganli":        (39.9283, 116.3728),
    "Xisi":            (39.9233, 116.3728),
    # Linha 10
    "Jinsong":         (39.8836, 116.4622),
    "Shuangjing":      (39.8886, 116.4722),
}

arestas_bj = [
    # Linha 1
    ("Sihui East",     "Sihui",           3),
    ("Sihui",          "Guomao",          3),
    ("Guomao",         "Yonganli",        3),
    ("Yonganli",       "Jianguomen",      3),
    ("Jianguomen",     "Dongdan",         3),
    ("Dongdan",        "Wangfujing",      3),
    ("Wangfujing",     "Tiananmen East",  3),
    ("Tiananmen East", "Tiananmen West",  3),
    ("Tiananmen West", "Xidan",           3),
    ("Xidan",          "Fuxingmen",       3),
    ("Fuxingmen",      "Muxidi",          3),
    # Linha 2
    ("Jianguomen",     "Chaoyangmen",     3),
    ("Chaoyangmen",    "Dongzhimen",      3),
    ("Dongzhimen",     "Andingmen",       3),
    ("Andingmen",      "Gulou Dajie",     3),
    ("Gulou Dajie",    "Jishuitan",       3),
    ("Jishuitan",      "Xizhimen",        3),
    ("Xizhimen",       "Fuchengmen",      3),
    ("Fuchengmen",     "Fuxingmen",       3),
    # Linha 4
    ("Xizhimen",       "Xinjiekou",       3),
    ("Xinjiekou",      "Pinganli",        3),
    ("Pinganli",       "Xisi",            3),
    ("Xisi",           "Xidan",           3),
    # Linha 10
    ("Guomao",         "Jinsong",         3),
    ("Jinsong",        "Shuangjing",      3),
]

grafo_bj = construir_grafo(arestas_bj)
print(f"Grafo Beijing: {len(grafo_bj)} estações, {sum(len(v) for v in grafo_bj.values())//2} arestas")

Grafo Beijing: 24 estações, 25 arestas


## 6. San Francisco — BART

**Origem:** Dublin/Pleasanton  
**Destino:** Daly City

### Caminhos distintos possíveis
1. Linha Azul: Dublin → Castro Valley → Bay Fair → San Leandro → ... → MacArthur → West Oakland → SF → Daly City
2. Linha Verde (bifurcação): Dublin → Castro Valley → Bay Fair → **Hayward** → MacArthur → West Oakland → SF → Daly City


In [6]:
coordenadas_bart = {
    "Dublin/Pleasanton": (37.7016, -121.8997),
    "Castro Valley":     (37.6940, -122.0758),
    "Bay Fair":          (37.6969, -122.1269),
    "Hayward":           (37.6703, -122.0878),
    "San Leandro":       (37.7022, -122.1608),
    "Fruitvale":         (37.7750, -122.2242),
    "Lake Merritt":      (37.7977, -122.2653),
    "12th St Oakland":   (37.8033, -122.2719),
    "19th St Oakland":   (37.8083, -122.2686),
    "MacArthur":         (37.8286, -122.2672),
    "West Oakland":      (37.8050, -122.2944),
    "Embarcadero":       (37.7928, -122.3967),
    "Montgomery":        (37.7894, -122.4014),
    "Powell":            (37.7844, -122.4078),
    "Civic Center":      (37.7794, -122.4144),
    "16th St Mission":   (37.7649, -122.4194),
    "24th St Mission":   (37.7522, -122.4183),
    "Glen Park":         (37.7330, -122.4339),
    "Balboa Park":       (37.7219, -122.4472),
    "Daly City":         (37.7064, -122.4686),
}

arestas_bart = [
    # Linha Azul
    ("Dublin/Pleasanton", "Castro Valley",    6),
    ("Castro Valley",     "Bay Fair",         5),
    ("Bay Fair",          "San Leandro",      4),
    ("San Leandro",       "Fruitvale",        4),
    ("Fruitvale",         "Lake Merritt",     4),
    ("Lake Merritt",      "12th St Oakland",  4),
    ("12th St Oakland",   "19th St Oakland",  3),
    ("19th St Oakland",   "MacArthur",        3),
    # Linha Verde (bifurcação em Bay Fair)
    ("Bay Fair",          "Hayward",          5),
    ("Hayward",           "MacArthur",        8),
    # Tronco compartilhado Oakland → SF → Daly City
    ("MacArthur",         "West Oakland",     5),
    ("West Oakland",      "Embarcadero",     10),
    ("Embarcadero",       "Montgomery",       3),
    ("Montgomery",        "Powell",           3),
    ("Powell",            "Civic Center",     3),
    ("Civic Center",      "16th St Mission",  4),
    ("16th St Mission",   "24th St Mission",  3),
    ("24th St Mission",   "Glen Park",        4),
    ("Glen Park",         "Balboa Park",      4),
    ("Balboa Park",       "Daly City",        4),
]

grafo_bart = construir_grafo(arestas_bart)
print(f"Grafo BART: {len(grafo_bart)} estações, {sum(len(v) for v in grafo_bart.values())//2} arestas")

Grafo BART: 20 estações, 20 arestas


## 7. Algoritmos de Busca

### 7.1 Caminho mais curto — Recursão com Memoização (`functools.lru_cache`)

Seguindo o esqueleto do checkpoint, usamos `lru_cache` para memoizar subproblemas.  
Como `lru_cache` exige argumentos *hashable*, o conjunto de visitados é passado como `frozenset`.

**Complexidade:**
- **Sem memo:** O(V!) no pior caso (explora todas as permutações de caminhos)
- **Com memo:** O(V · 2^V) — cada par `(nó, visitados)` é calculado uma única vez


In [7]:
def menor_custo_com_memo(grafo, origem, destino, hora):
    """
    Caminho mais curto com memoização via lru_cache.
    Converte o grafo para tuplas (necessário para lru_cache hashable).
    """
    # lru_cache exige que todos os argumentos sejam hashable
    grafo_tuple = {k: tuple(v) for k, v in grafo.items()}

    @functools.lru_cache(maxsize=None)
    def _dp(atual, visitados):
        if atual == destino:
            return 0
        melhor = float('inf')
        for vizinho, peso in grafo_tuple.get(atual, ()):
            if vizinho not in visitados:
                fator = fator_horario(hora)
                custo = fator * peso + _dp(vizinho, visitados | frozenset([atual]))
                melhor = min(melhor, custo)
        return melhor

    _dp.cache_clear()
    return _dp(origem, frozenset())


def menor_custo_sem_memo(grafo, origem, destino, hora, visitados=None):
    """Caminho mais curto — recursão pura, sem memoização (para comparação)."""
    if visitados is None:
        visitados = set()
    if origem == destino:
        return 0
    visitados.add(origem)
    melhor = float('inf')
    for vizinho, peso in grafo.get(origem, []):
        if vizinho not in visitados:
            custo = fator_horario(hora) * peso + menor_custo_sem_memo(
                grafo, vizinho, destino, hora, visitados)
            melhor = min(melhor, custo)
    visitados.remove(origem)
    return melhor

### 7.2 Caminho mais longo simples — Backtracking

O caminho mais longo *simples* (sem ciclos) não admite memoização clássica, pois o resultado 
depende do caminho percorrido até o nó atual (conjunto de visitados). Usamos **backtracking** puro.

**Complexidade:** O(V!) — explora todos os caminhos sem ciclos.


In [8]:
def maior_custo(grafo, origem, destino, hora, visitados=None):
    """Caminho mais longo simples — backtracking sem ciclos."""
    if visitados is None:
        visitados = set()
    if origem == destino:
        return 0
    visitados.add(origem)
    melhor = float('-inf')
    for vizinho, peso in grafo.get(origem, []):
        if vizinho not in visitados:
            custo = fator_horario(hora) * peso + maior_custo(
                grafo, vizinho, destino, hora, visitados)
            if custo > melhor:
                melhor = custo
    visitados.remove(origem)
    return melhor

### 7.3 Reconstrução dos caminhos

In [9]:
def reconstruir_curto(grafo, origem, destino, hora, visitados=None):
    """Retorna (custo, [caminho]) do menor custo."""
    if visitados is None:
        visitados = set()
    if origem == destino:
        return (0, [destino])
    visitados.add(origem)
    melhor = None
    for vizinho, peso in grafo.get(origem, []):
        if vizinho not in visitados:
            res = reconstruir_curto(grafo, vizinho, destino, hora, visitados)
            if res is not None:
                total = fator_horario(hora) * peso + res[0]
                if melhor is None or total < melhor[0]:
                    melhor = (total, [origem] + res[1])
    visitados.remove(origem)
    return melhor


def reconstruir_longo(grafo, origem, destino, hora, visitados=None):
    """Retorna (custo, [caminho]) do maior custo."""
    if visitados is None:
        visitados = set()
    if origem == destino:
        return (0, [destino])
    visitados.add(origem)
    melhor = None
    for vizinho, peso in grafo.get(origem, []):
        if vizinho not in visitados:
            res = reconstruir_longo(grafo, vizinho, destino, hora, visitados)
            if res is not None:
                total = fator_horario(hora) * peso + res[0]
                if melhor is None or total > melhor[0]:
                    melhor = (total, [origem] + res[1])
    visitados.remove(origem)
    return melhor

## 8. Análise de Desempenho

Para cada cidade medimos **tempo de execução** (`time.perf_counter`) e **uso de memória** (`tracemalloc`) 
com e sem memoização, exibindo os resultados em tabela comparativa.


In [10]:
def analisar_cidade(nome, grafo, coordenadas, arestas, origem, destino, hora):
    print(f"\n{'═'*68}")
    print(f"  {nome}  |  {origem} → {destino}  |  Horário: {hora}h  (fator {fator_horario(hora)}x)")
    print(f"{'═'*68}")
    print(f"{'Algoritmo':<35} {'Custo':>7} {'Tempo(s)':>10} {'Mem(KB)':>10}")
    print(f"{'-'*68}")

    # — com memoização —
    tracemalloc.start()
    t0 = time.perf_counter()
    custo_memo = menor_custo_com_memo(grafo, origem, destino, hora)
    t1 = time.perf_counter()
    mem_memo = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()

    # — sem memoização —
    tracemalloc.start()
    t2 = time.perf_counter()
    custo_sem = menor_custo_sem_memo(grafo, origem, destino, hora)
    t3 = time.perf_counter()
    mem_sem = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()

    # — caminho mais longo (backtracking) —
    tracemalloc.start()
    t4 = time.perf_counter()
    custo_longo = maior_custo(grafo, origem, destino, hora)
    t5 = time.perf_counter()
    mem_longo = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()

    print(f"{'Mais curto (com memoização)':<35} {custo_memo:>7.1f} {t1-t0:>10.4f} {mem_memo/1024:>10.1f}")
    print(f"{'Mais curto (sem memoização)':<35} {custo_sem:>7.1f} {t3-t2:>10.4f} {mem_sem/1024:>10.1f}")
    print(f"{'Mais longo (backtracking)':<35} {custo_longo:>7.1f} {t5-t4:>10.4f} {mem_longo/1024:>10.1f}")

    res_curto = reconstruir_curto(grafo, origem, destino, hora)
    res_longo  = reconstruir_longo(grafo, origem, destino, hora)

    caminho_curto = res_curto[1] if res_curto else []
    caminho_longo  = res_longo[1]  if res_longo  else []

    print(f"\nCaminho mais curto ({len(caminho_curto)} estações):")
    print("  " + " → ".join(caminho_curto))
    print(f"\nCaminho mais longo ({len(caminho_longo)} estações):")
    print("  " + " → ".join(caminho_longo))

    # — mapa folium —
    centro = list(coordenadas[origem])
    mapa = folium.Map(location=centro, zoom_start=12)

    for o, d, _ in arestas:
        if o in coordenadas and d in coordenadas:
            folium.PolyLine(
                [coordenadas[o], coordenadas[d]],
                color="gray", weight=2, opacity=0.5
            ).add_to(mapa)

    for est, coord in coordenadas.items():
        folium.CircleMarker(
            coord, radius=5, color="blue",
            fill=True, fill_color="blue", tooltip=est
        ).add_to(mapa)

    for i in range(len(caminho_curto) - 1):
        folium.PolyLine(
            [coordenadas[caminho_curto[i]], coordenadas[caminho_curto[i+1]]],
            color="red", weight=5,
            tooltip=f"{caminho_curto[i]} → {caminho_curto[i+1]}"
        ).add_to(mapa)

    folium.Marker(coordenadas[origem], popup=f"ORIGEM: {origem}",
                  icon=folium.Icon(color="green", icon="play")).add_to(mapa)
    folium.Marker(coordenadas[destino], popup=f"DESTINO: {destino}",
                  icon=folium.Icon(color="red", icon="stop")).add_to(mapa)

    arquivo = f"mapa_{nome.lower().replace(' ', '_').replace('(','').replace(')','')}.html"
    mapa.save(arquivo)
    print(f"\nMapa salvo: {arquivo}")
    return mapa

## 9. Execução — Análise das 3 Cidades

Informe o horário de partida para que os fatores de penalidade/bônus sejam aplicados.


In [11]:
hora = int(input("Informe o horário de partida (0-23): "))

mapa_sp   = analisar_cidade(
    "São Paulo", grafo_sp, coordenadas_sp, arestas_sp,
    "Tucuruvi", "Capão Redondo", hora)

mapa_bj   = analisar_cidade(
    "Beijing", grafo_bj, coordenadas_bj, arestas_bj,
    "Sihui East", "Xizhimen", hora)

mapa_bart = analisar_cidade(
    "San Francisco (BART)", grafo_bart, coordenadas_bart, arestas_bart,
    "Dublin/Pleasanton", "Daly City", hora)


════════════════════════════════════════════════════════════════════
  São Paulo  |  Tucuruvi → Capão Redondo  |  Horário: 15h  (fator 1.0x)
════════════════════════════════════════════════════════════════════
Algoritmo                             Custo   Tempo(s)    Mem(KB)
--------------------------------------------------------------------
Mais curto (com memoização)            90.0     0.0036      307.8
Mais curto (sem memoização)            90.0     0.0010        4.5
Mais longo (backtracking)             105.0     0.0006        3.4

Caminho mais curto (31 estações):
  Tucuruvi → Parada Inglesa → Jardim São Paulo → Carandiru → Santana → Portuguesa-Tietê → Armênia → Tiradentes → Luz → São Bento → Sé → Japão-Liberdade → Vergueiro → Paraíso → Ana Rosa → Santa Cruz → Hospital São Paulo → AACD-Servidor → Moema → Eucaliptos → Campo Belo → Brooklin → Borba Gato → Alto da Boa Vista → Adolfo Pinheiro → Largo Treze → Santo Amaro → Giovanni Gronchi → Vila das Belezas → Campo Limpo → Capão Re

## 10. Conclusões

### Memoização vs. Sem Memoização
A memoização com `functools.lru_cache` evita recomputar subproblemas já resolvidos.  
Em grafos com múltiplos caminhos, a versão sem memo revisita os mesmos estados repetidamente, 
resultando em tempo exponencial. Com memo, cada estado `(nó, visitados)` é calculado **uma única vez**.

### Caminho mais longo
O caminho mais longo simples (sem ciclos) **não pode ser memoizado** de forma eficiente, 
pois o custo ótimo a partir de um nó depende do histórico de visitados — o que torna o 
espaço de estados exponencial de qualquer forma. Por isso usamos backtracking puro.

### Fatores de horário
O fator multiplicador altera os custos absolutamente, mas **não muda o caminho ótimo** 
em grafos com pesos uniformes — a ordenação relativa das rotas permanece igual.  
Em grafos com pesos variados (como o BART), o fator pode mudar a rota preferida.

### Complexidade Big-O
| Algoritmo | Complexidade |
|---|---|
| Recursão sem memo | O(V!) |
| Recursão com memo (lru_cache) | O(V · 2^V) |
| Backtracking (mais longo) | O(V!) |

onde **V** = número de vértices (estações).
